In [27]:
"""
Gibson-Schwartz DC-PINN | Phase 1 skeleton
==========================================
Networks + hard-wired pricing transform + global params + smoke test.
NO losses, NO training yet.

Backbones (Dense, MLP, weight_fact reparam) are ported near-verbatim from the
DC-PINN reference. The one deliberate change: the softplus on the MLP output is
REMOVED. The reference forced positive output (implied vol); convenience yield
must admit negatives, so both heads use identity output.
"""
from typing import Callable, Dict, Tuple, Union
from dataclasses import dataclass

import jax
import jax.numpy as jnp
from jax import random
from jax.nn.initializers import glorot_normal, normal, zeros
from flax import linen as nn
import optax
import json
import pickle
from pathlib import Path


REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_DIR = REPO_ROOT / "config"
MC_DATA_PATH = REPO_ROOT / "data" / "input" / "synthetic" / "mc_data.pkl"
CKPT_DIR = REPO_ROOT / "data" / "output" / "checkpoints"   # gitignored, disposable

## Network Architecture

Ported from the original DC-PINN repo, but with a few changes:
•  The original repo used a single network to predict both the drift and diffusion terms. Here we use two separate networks, one for each term.

In [28]:
_ACT = {
        "tanh": jnp.tanh, 
        "sin": jnp.sin
        }

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_DIR = REPO_ROOT / "config"


def _weight_fact(init_fn, mean, stddev):
    def init(key, shape):
        k1, k2 = random.split(key)
        w = init_fn(k1, shape)
        g = jnp.exp(mean + normal(stddev)(k2, (shape[-1],)))
        return g, w / g
    return init


class Dense(nn.Module):
    features: int
    kernel_init: Callable = glorot_normal()
    bias_init: Callable = zeros
    reparam: Union[None, Dict] = None

    @nn.compact
    def __call__(self, x):
        if self.reparam is None:
            kernel = self.param("kernel", self.kernel_init,
                                (x.shape[-1], self.features))
        elif self.reparam["type"] == "weight_fact":
            g, v = self.param(
                "kernel",
                _weight_fact(self.kernel_init,
                             self.reparam["mean"], self.reparam["stddev"]),
                (x.shape[-1], self.features))
            kernel = g * v
        bias = self.param("bias", self.bias_init, (self.features,))
        return jnp.dot(x, kernel) + bias


class MLP(nn.Module):
    hidden_dim: Tuple[int, ...] = (16, 16, 16, 16)
    out_dim: int = 1
    activation: str = "tanh"
    reparam: Union[None, Dict] = None

    @nn.compact
    def __call__(self, x):
        act = _ACT[self.activation]
        for h in self.hidden_dim:
            x = act(Dense(h, reparam=self.reparam)(x))
        x = Dense(self.out_dim, reparam=self.reparam)(x)
        return x                      


## Parameter Initialisation

In [29]:
def _softplus(x):     
    return jnp.logaddexp(x, 0.0)

def _inv_softplus(y): 
    return jnp.log(jnp.expm1(y))     # softplus^{-1}


def load_global_params_from_json(p_path=None, q_path=None):
    p_path = Path(p_path) if p_path is not None else CONFIG_DIR / "p_params.json"
    q_path = Path(q_path) if q_path is not None else CONFIG_DIR / "q_params.json"

    with open(p_path, "r") as f:
        cfg = json.load(f)
    with open(q_path, "r") as f:
        cfg.update(json.load(f))     # q keys (r, lambda2) join the p keys

    required = ("kappa", "alpha_P", "sigma1", "sigma2", "rho", "lambda2")
    missing = [k for k in required if k not in cfg]
    if missing:
        raise KeyError(f"missing keys {missing} in {p_path.name} / {q_path.name}; "
                       f"found {sorted(cfg)}")


    for k in ("kappa", "sigma1", "sigma2"):
        if cfg[k] <= 0.0:
            raise ValueError(f"{k} must be > 0 for the softplus reparam, got {cfg[k]}")
    if not -1.0 < cfg["rho"] < 1.0:
        raise ValueError(f"rho must lie strictly in (-1, 1), got {cfg['rho']}")

    return {
        "kappa_raw":  _inv_softplus(jnp.array(cfg["kappa"])),
        "sigma1_raw": _inv_softplus(jnp.array(cfg["sigma1"])),
        "sigma2_raw": _inv_softplus(jnp.array(cfg["sigma2"])),
        "alpha_P":    jnp.array(cfg["alpha_P"]),
        "lambda2":    jnp.array(cfg["lambda2"]),
        "rho_raw":    jnp.arctanh(jnp.array(cfg["rho"])),
    }


def init_global_params():
    """Raw params whose constrained values equal the defaults below."""
    return {
        "kappa_raw":  _inv_softplus(jnp.array(1.0)),   # kappa  > 0
        "sigma1_raw": _inv_softplus(jnp.array(0.3)),   # sigma1 > 0  (spot vol)
        "sigma2_raw": _inv_softplus(jnp.array(0.3)),   # sigma2 > 0  (conv-yield vol)
        "alpha_P":    jnp.array(0.0),                  # free (can be negative)
        "lambda2":    jnp.array(0.0),                  # free
        "rho_raw":    jnp.array(0.0),                  # rho = tanh(raw) in (-1,1)
    }


def constrain(p):
    return {
        "kappa":   _softplus(p["kappa_raw"]),
        "sigma1":  _softplus(p["sigma1_raw"]),
        "sigma2":  _softplus(p["sigma2_raw"]),
        "alpha_P": p["alpha_P"],
        "lambda2": p["lambda2"],
        "rho":     jnp.tanh(p["rho_raw"]),
    }


def alpha_Q(c):
    """Derived, NOT free:  alpha^Q = alpha^P - sigma2 * lambda2 / kappa."""
    return c["alpha_P"] - c["sigma2"] * c["lambda2"] / c["kappa"]

In [30]:
# ---------------------------------------------------------------------------
# Hard-wired pieces
# ---------------------------------------------------------------------------
def B_tau(tau, kappa):
    """Analytic delta-channel.  B(tau) = -(1 - e^{-kappa tau}) / kappa."""
    return -(1.0 - jnp.exp(-kappa * tau)) / kappa


def log_price(S, delta, tau, A, kappa):
    """log F_hat = log S + B(tau) * delta + A.

    The affine form lives HERE rather than inside price(): the observable is
    log_F_obs, so every loss works in log space. Going through exp() only to
    take log() again adds round-trip error and can overflow while A is still
    unconstrained early in training. price() is defined as the exp of this, so
    there remains exactly one definition of the pricing map."""
    return jnp.log(S) + B_tau(tau, kappa) * delta + A


def price(S, delta, tau, A, kappa):
    """F_hat = S * exp( B(tau) * delta + A )."""
    return jnp.exp(log_price(S, delta, tau, A, kappa))


# ---------------------------------------------------------------------------
# Forward assembly
# ---------------------------------------------------------------------------
def forward_log(net_d, net_A, params_d, params_A, gp_raw, t, S, tau):
    """Log-space forward — this is what the losses call.
    t, S, tau : shape (N,) arrays.
    Returns log_F_hat, delta_hat, A_hat  (each (N,)).
    """
    c = constrain(gp_raw)
    delta = net_d.apply(params_d, t[:, None])[:, 0]
    A     = net_A.apply(params_A, tau[:, None])[:, 0]
    logF  = log_price(S, delta, tau, A, c["kappa"])
    return logF, delta, A


def forward(net_d, net_A, params_d, params_A, gp_raw, t, S, tau):
    """Level-space forward. Convenience wrapper for inspection/plotting;
    losses should use forward_log to stay in the observable's own space.
    Returns F_hat, delta_hat, A_hat  (each (N,)).
    """
    logF, delta, A = forward_log(net_d, net_A, params_d, params_A,
                                 gp_raw, t, S, tau)
    return jnp.exp(logF), delta, A


def make_models(hidden=(16, 16, 16, 16), activation="tanh"):
    net_d = MLP(hidden, 1, activation)     # N_delta : t   -> delta
    net_A = MLP(hidden, 1, activation)     # N_A     : tau -> A
    return net_d, net_A


def init_all(seed=0, hidden=(16, 16, 16, 16), activation="tanh", gp_from_json=False):
    net_d, net_A = make_models(hidden, activation)
    key = random.PRNGKey(seed)
    k_d, k_A = random.split(key)           # one split per net, no key reuse
    params_d = net_d.init(k_d, jnp.ones((1, 1)))
    params_A = net_A.init(k_A, jnp.ones((1, 1)))
    # gp_from_json=True starts the globals AT the ground truth — only valid for a
    # sanity/oracle run. For the actual inversion experiment leave it False, or
    # the "recovered" parameters are just the ones you handed the optimiser.
    gp_raw = load_global_params_from_json() if gp_from_json else init_global_params()
    return net_d, net_A, params_d, params_A, gp_raw

In [31]:
# ---------------------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    net_d, net_A, params_d, params_A, gp_raw = init_all(seed=0)

    N = 8
    key = random.PRNGKey(42)
    k1, k2, k3 = random.split(key, 3)
    t   = random.uniform(k1, (N,), minval=0.0, maxval=5.0)          # calendar time
    S   = random.uniform(k2, (N,), minval=40.0, maxval=90.0)        # spot
    tau = random.uniform(k3, (N,), minval=0.01, maxval=2.0)         # maturity

    F, delta, A = forward(net_d, net_A, params_d, params_A, gp_raw, t, S, tau)
    c = constrain(gp_raw)

    print("=== constrained global params (defaults) ===")
    for k, v in c.items():
        print(f"  {k:8s} = {float(v): .4f}")
    print(f"  alpha_Q  = {float(alpha_Q(c)): .4f}   (derived)")

    print("\n=== forward pass (N=8) ===")
    print("  shapes :", F.shape, delta.shape, A.shape)
    print("  delta  :", jnp.round(delta, 4))
    print("  A      :", jnp.round(A, 4))
    print("  F_hat  :", jnp.round(F, 4))

    assert F.shape == delta.shape == A.shape == (N,)
    assert jnp.all(jnp.isfinite(F)), "non-finite F"
    assert jnp.all(jnp.isfinite(delta)), "non-finite delta"

    # IC check at tau->0 : B(0)=0 so F(0)=S*exp(A(0)); untrained A(0)!=0 so F!=S.
    # This is EXPECTED — the IC A(0)=0 is left to the A-ODE, not hard-wired.
    tau0 = jnp.full((N,), 1e-6)
    F0, _, A0 = forward(net_d, net_A, params_d, params_A, gp_raw, t, S, tau0)
    print("\n=== IC sanity (tau -> 0) ===")
    print("  A(~0) :", jnp.round(A0, 4))
    print("  |F0 - S| mean :", float(jnp.mean(jnp.abs(F0 - S))),
          "  (nonzero pre-training = correct)")

    # -----------------------------------------------------------------------
    # JSON loader: the raw <-> constrained transforms must round-trip, i.e. the
    # constrained values seen by the model equal the numbers on disk. A silent
    # rename or a dropped inverse-transform shows up here as a mismatch rather
    # than as a plausible-looking but wrong parameter thousands of steps later.
    # -----------------------------------------------------------------------
    with open(CONFIG_DIR / "p_params.json") as f:
        cfg_disk = json.load(f)
    with open(CONFIG_DIR / "q_params.json") as f:
        cfg_disk.update(json.load(f))

    c_json = constrain(load_global_params_from_json())

    print("\n=== JSON round-trip (config/{p,q}_params.json) ===")
    for k in ("kappa", "alpha_P", "sigma1", "sigma2", "rho", "lambda2"):
        print(f"  {k:8s} = {float(c_json[k]): .6f}   (disk {cfg_disk[k]: .6f})")
        assert jnp.allclose(c_json[k], cfg_disk[k], atol=1e-5), \
            f"{k} did not round-trip: {float(c_json[k])} vs {cfg_disk[k]}"

    # alpha_Q must agree with the same formula monte_carlo_simulation.ipynb uses
    # to price the panel — if these two drift apart the PINN is inverting against
    # data generated under different Q dynamics than it assumes.
    aQ_mc = (cfg_disk["alpha_P"]
             - cfg_disk["sigma2"] * cfg_disk["lambda2"] / cfg_disk["kappa"])
    print(f"  alpha_Q  = {float(alpha_Q(c_json)): .6f}   (MC notebook {aQ_mc: .6f})")
    assert jnp.allclose(alpha_Q(c_json), aQ_mc, atol=1e-6), "alpha_Q disagrees with MC"

    # Loading the truth must actually move the params off the defaults, else the
    # gp_from_json switch is a no-op and this whole check proves nothing.
    assert not jnp.allclose(c_json["kappa"], c["kappa"]), "JSON load had no effect"

    # The forward pass must survive the ground-truth params too, not just the
    # benign defaults: kappa=1.876 makes B(tau) smaller, exp() argument larger.
    F_json, delta_json, _ = forward(net_d, net_A, params_d, params_A,
                                    load_global_params_from_json(), t, S, tau)
    assert jnp.all(jnp.isfinite(F_json)), "non-finite F under JSON params"

    print("\nSMOKE TEST PASSED")

=== constrained global params (defaults) ===
  kappa    =  1.0000
  sigma1   =  0.3000
  sigma2   =  0.3000
  alpha_P  =  0.0000
  lambda2  =  0.0000
  rho      =  0.0000
  alpha_Q  =  0.0000   (derived)

=== forward pass (N=8) ===
  shapes : (8,) (8,) (8,)
  delta  : [0.44009998 0.4886     0.3265     0.3847     0.4109     0.4096
 0.3786     0.4579    ]
  A      : [-0.3773 -0.3821 -0.1212 -0.2837 -0.3153 -0.0443 -0.0361 -0.2658]
  F_hat  : [37.8575   37.2915   40.3266   32.8715   26.265999 46.2351   52.4188
 27.8809  ]

=== IC sanity (tau -> 0) ===
  A(~0) : [-0. -0. -0. -0. -0. -0. -0. -0.]
  |F0 - S| mean : 5.2928924560546875e-05   (nonzero pre-training = correct)

=== JSON round-trip (config/{p,q}_params.json) ===
  kappa    =  1.876000   (disk  1.876000)
  alpha_P  =  0.106000   (disk  0.106000)
  sigma1   =  0.393000   (disk  0.393000)
  sigma2   =  0.527000   (disk  0.527000)
  rho      =  0.766000   (disk  0.766000)
  lambda2  =  0.100000   (disk  0.100000)
  alpha_Q  =  0.07790

## Phase 3 — Training Harness

Generic, loss-**agnostic** training loop, parameterised by a `loss_fn` so the
Phase 4 data-misfit (and Phase 5/6 terms) slot in without touching this code.

Key design points:

* **One optimiser** over the combined pytree `{net_d, net_A, gp}`.
* **Globals are freezable** (`multi_transform` + `set_to_zero`). For the Phase 4
  gate they are frozen so the gate tests *only* "can the nets fit δ from the
  panel", isolated from parameter identification (Phase 8).
* **Per-path batching** — the architecture inverts a single realisation, so the
  harness selects `path_idx` and flattens that one path's panel.
* The in-loop eval metric **is** the gate criterion: shape-correlation of
  `delta_hat` vs `delta_true` (should recover) plus level offset (allowed to be
  nonzero under MSE-only, since the level is confounded until the NLL / A-ODE).

In [32]:
@dataclass
class TrainConfig:
    epochs: int = 20000
    lr: float = 1e-3
    decay_rate: float = 0.9
    decay_steps: int = 2000
    log_every: int = 500
    ckpt_every: int = 5000                    # 0 disables checkpointing
    ckpt_path: str = str(CKPT_DIR / "phase_ckpt.pkl")
    path_idx: int = 0                         # which synthetic path to invert
    train_globals: bool = False               # Phase 4 gate: False = freeze globals
    # When globals are FROZEN they must be frozen at the TRUE values, not at the
    # init_global_params() defaults: kappa enters B(tau) = -(1-e^{-kappa tau})/kappa,
    # so freezing at kappa=1.0 when the data was generated with kappa=1.876 means
    # the delta-channel is misspecified and the gate would be asking the net to fit
    # delta through the wrong transform. Set False only to study that misspecification.
    gp_from_json: bool = True
    seed: int = 0


# ---------------------------------------------------------------------------
# Data plumbing
# ---------------------------------------------------------------------------
def load_mc_data(path=None):
    """Load the Phase-2 synthetic dataset written by monte_carlo_simulation.ipynb."""
    with open(path or MC_DATA_PATH, "rb") as f:
        return pickle.load(f)


def make_batch(mc_data, path_idx):
    """Flatten ONE path's panel into (P,) rows, P=(N+1)*K.
       Returns (batch, eval_) with eval arrays kept separate from training rows.
       Each row is one (date, maturity) observation; S and t repeat across
       maturities, tau tiles across dates."""
    t_grid = jnp.asarray(mc_data["t_grid"])              # (N+1,)
    taus   = jnp.asarray(mc_data["taus"])                # (K,)
    S_path = jnp.asarray(mc_data["S"][path_idx])         # (N+1,)
    logF   = jnp.asarray(mc_data["log_F_obs"][path_idx]) # (N+1, K)
    Np1, K = logF.shape

    batch = {
        "t":   jnp.repeat(t_grid, K),           # date varies slow
        "S":   jnp.repeat(S_path, K),
        "tau": jnp.tile(taus, Np1),             # maturity varies fast
        "logF": logF.reshape(-1),               # target for L_data
    }
    # delta_true is EVALUATION ONLY — it is deliberately kept out of `batch` so it
    # cannot reach a loss function by accident (CLAUDE.md sec. 6.3).
    eval_ = {"t_grid": t_grid,
             "delta_true": jnp.asarray(mc_data["delta_true"][path_idx])}
    return batch, eval_


def make_params(cfg, hidden=(16, 16, 16, 16), activation="tanh"):
    """Init nets + globals. Globals start at the JSON truth (cfg.gp_from_json=True,
       the frozen-at-truth gate) or at init_global_params() (deliberately wrong)."""
    net_d, net_A = make_models(hidden, activation)
    kd, kA = jax.random.split(jax.random.PRNGKey(cfg.seed))
    params = {
        "net_d": net_d.init(kd, jnp.ones((1, 1))),
        "net_A": net_A.init(kA, jnp.ones((1, 1))),
        "gp":    load_global_params_from_json() if cfg.gp_from_json
                 else init_global_params(),
    }
    return net_d, net_A, params


# ---------------------------------------------------------------------------
# Optimizer with freezable globals
# ---------------------------------------------------------------------------
def make_optimizer(cfg, params):
    sched = optax.exponential_decay(cfg.lr, cfg.decay_steps, cfg.decay_rate)
    base = optax.adam(sched)
    if cfg.train_globals:
        return base
    # freeze gp leaves -> set_to_zero; train the two nets
    labels = {
        "net_d": jax.tree_util.tree_map(lambda _: "train", params["net_d"]),
        "net_A": jax.tree_util.tree_map(lambda _: "train", params["net_A"]),
        "gp":    jax.tree_util.tree_map(lambda _: "frozen", params["gp"]),
    }
    return optax.multi_transform(
        {"train": base, "frozen": optax.set_to_zero()}, labels)


def make_train_step(loss_fn, optimizer):
    @jax.jit
    def step(params, opt_state, batch):
        (loss, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(params, batch)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss, aux
    return step


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------
def train(loss_fn, params, batch, eval_, cfg, net_d):
    optimizer = make_optimizer(cfg, params)
    opt_state = optimizer.init(params)
    step = make_train_step(loss_fn, optimizer)

    dt_true = eval_["delta_true"]
    t_col = eval_["t_grid"][:, None]
    history = []
    for epoch in range(cfg.epochs):
        params, opt_state, loss, aux = step(params, opt_state, batch)

        if epoch % cfg.log_every == 0:
            dh = net_d.apply(params["net_d"], t_col)[:, 0]        # delta_hat on the date grid
            corr = (float(jnp.corrcoef(dh, dt_true)[0, 1])
                    if float(jnp.std(dh)) > 1e-8 else float("nan"))
            off = float(jnp.mean(dh - dt_true))                  # level offset (allowed nonzero)
            history.append((epoch, float(loss), corr, off))
            print(f"epoch {epoch:6d} | loss {float(loss):.4e} | "
                  f"shape-corr {corr:+.3f} | level-off {off:+.4f}")

        if cfg.ckpt_every and epoch and epoch % cfg.ckpt_every == 0:
            Path(cfg.ckpt_path).parent.mkdir(parents=True, exist_ok=True)
            with open(cfg.ckpt_path, "wb") as f:
                pickle.dump({"params": params, "epoch": epoch}, f)

    return params, history


# ---------------------------------------------------------------------------
# Phase 4 — L_data
# ---------------------------------------------------------------------------
def l_data_factory(net_d, net_A):
    """MSE between predicted and observed LOG futures prices.

        L_data = (1/P) * sum_p ( log F_hat_p - log F_obs_p )^2

    Log space, not level space, for three reasons:
      1. The observable IS log_F_obs — the noise was added additively in log
         space by the observation layer, so squared error in log space is the
         correctly-specified Gaussian likelihood. Level-space MSE would instead
         imply additive noise on F, which is not the DGP.
      2. F ranges ~[10, 550] across the panel, so level-space MSE weights long
         maturities and high-spot dates far more heavily; log space is scale-free
         and weights every observation equally.
      3. log F_hat = log S + B(tau)*delta + A is affine and exact — no exp/log
         round trip, and no overflow risk while A is unconstrained.

    Returned aux carries RMSE in log units, directly comparable to the
    generator's noise_std (0.01): converging below it means fitting noise.
    """
    def loss(params, batch):
        logF_hat, delta, A = forward_log(
            net_d, net_A, params["net_d"], params["net_A"], params["gp"],
            batch["t"], batch["S"], batch["tau"])
        resid = logF_hat - batch["logF"]          # (P,) in log-price units
        mse = jnp.mean(resid ** 2)
        return mse, {"rmse_log": jnp.sqrt(mse),
                     "delta_mean": jnp.mean(delta),
                     "A_mean": jnp.mean(A)}
    return loss


# ---------------------------------------------------------------------------
# HARNESS-TEST SCAFFOLDING ONLY — kept to isolate plumbing from the real loss.
# ---------------------------------------------------------------------------
def _dummy_loss_factory(net_d, net_A):
    """NOT L_data. Drives net outputs toward zero to exercise gradient flow
       through both nets and the (frozen) globals path. Consumes the batch via
       the real forward so batching/plumbing is genuinely tested."""
    def loss(params, batch):
        F, delta, A = forward(net_d, net_A,
                              params["net_d"], params["net_A"], params["gp"],
                              batch["t"], batch["S"], batch["tau"])
        return jnp.mean(delta ** 2) + jnp.mean(A ** 2), {"mean_F": jnp.mean(F)}
    return loss

In [33]:
# ---------------------------------------------------------------------------
# Phase 4 gate — L_data on the REAL mc_data.pkl.
# Exercises: pickle schema, batching, gradient flow, global freezing, ckpt IO,
# then asks the actual question: can the nets recover delta from the panel with
# the globals frozen at truth?
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    mc_data = load_mc_data()
    M, Np1 = mc_data["delta_true"].shape
    K = mc_data["taus"].shape[0]

    print("=== mc_data.pkl ===")
    print(f"  paths M       : {M}")
    print(f"  dates N+1     : {Np1}   (T = {mc_data['T']}, N = {mc_data['N']})")
    print(f"  maturities K  : {K}     (tau {float(mc_data['taus'].min()):.3f} "
          f"-> {float(mc_data['taus'].max()):.3f})")
    print(f"  noise_std     : {mc_data['noise_std']}")

    # The pickle carries the params it was generated with. If these disagree with
    # config/*.json the notebook would be inverting against data from a different
    # DGP than gp_from_json loads — a silent, invisible mismatch.
    with open(CONFIG_DIR / "p_params.json") as f:
        cfg_disk = json.load(f)
    with open(CONFIG_DIR / "q_params.json") as f:
        cfg_disk.update(json.load(f))
    for k, v in mc_data["params_P"].items():
        assert jnp.allclose(v, cfg_disk[k]), \
            f"pickle params_P[{k}]={v} != config {cfg_disk[k]} — regenerate mc_data.pkl"
    assert jnp.allclose(mc_data["lambda2"], cfg_disk["lambda2"]), "lambda2 mismatch"
    assert jnp.allclose(mc_data["alpha_Q"], alpha_Q(constrain(
        load_global_params_from_json())), atol=1e-6), "alpha_Q mismatch"
    print("  params_P / lambda2 / alpha_Q all match config/*.json  ✓")

    cfg = TrainConfig(epochs=5000, log_every=500, ckpt_every=0, path_idx=0)
    batch, eval_ = make_batch(mc_data, cfg.path_idx)
    net_d, net_A, params = make_params(cfg)

    P = Np1 * K
    print(f"\n=== batch (path {cfg.path_idx}) ===")
    for k, v in batch.items():
        print(f"  {k:5s} {str(v.shape):10s} "
              f"[{float(v.min()): .4f}, {float(v.max()): .4f}]")
    assert all(batch[k].shape == (P,) for k in ("t", "S", "tau", "logF"))
    assert "delta_true" not in batch, "ground truth leaked into the training batch"

    # tile/repeat convention: row p = (date p//K, maturity p%K). Check the seam —
    # a transposed reshape would still give the right shapes but wrong pairings.
    assert jnp.allclose(batch["tau"][:K], mc_data["taus"]), "tau not tiling fastest"
    assert jnp.allclose(batch["t"][:K], mc_data["t_grid"][0]), "t not repeating slowest"
    assert jnp.allclose(batch["logF"][:K], mc_data["log_F_obs"][cfg.path_idx, 0]), \
        "logF flattening does not match (date, maturity) row order"
    print("  row order (date varies slow, maturity fast) verified  ✓")

    # log_price must reproduce the generator's own pricing when handed the TRUE
    # state and the TRUE params. This is the strongest available check that the
    # forward map matches the DGP — if it fails, no amount of training can help.
    c_true = constrain(load_global_params_from_json())
    d_true_rows = jnp.repeat(jnp.asarray(mc_data["delta_true"][cfg.path_idx]), K)
    A_true = (jnp.log(jnp.asarray(mc_data["F_clean"][cfg.path_idx]).reshape(-1))
              - jnp.log(batch["S"])
              - B_tau(batch["tau"], c_true["kappa"]) * d_true_rows)   # implied A
    logF_oracle = log_price(batch["S"], d_true_rows, batch["tau"],
                            A_true, c_true["kappa"])
    orac = float(jnp.sqrt(jnp.mean(
        (logF_oracle - jnp.log(jnp.asarray(mc_data["F_clean"][cfg.path_idx]).reshape(-1))) ** 2)))
    assert orac < 1e-5, f"log_price does not reproduce F_clean, rmse {orac}"
    # A(tau) implied from clean prices must depend on tau ONLY (it is a function of
    # tau alone in the closed form). Its spread across dates bounds how well any
    # tau-only net_A can possibly fit.
    A_bydate = A_true.reshape(Np1, K)
    print(f"  log_price vs F_clean rmse : {orac:.2e}  ✓")
    print(f"  implied A(tau) spread across dates : "
          f"{float(jnp.max(jnp.std(A_bydate, axis=0))):.2e}  (should be ~0)")

    gp0 = jax.tree_util.tree_map(lambda x: x.copy(), params["gp"])   # snapshot frozen globals

    print(f"\n=== training: L_data (log-space MSE), globals frozen at truth ===")
    print(f"  target noise floor: rmse_log = noise_std = {mc_data['noise_std']}")
    loss_fn = l_data_factory(net_d, net_A)
    params, hist = train(loss_fn, params, batch, eval_, cfg, net_d)

    # 1) loss decreased -> gradients flow, optimizer steps
    assert hist[-1][1] < hist[0][1], "L_data did not decrease"
    # 2) frozen globals did NOT move
    moved = max(float(jnp.max(jnp.abs(params["gp"][k] - gp0[k]))) for k in gp0)
    assert moved == 0.0, f"frozen globals moved by {moved}"
    # 3) globals were frozen AT THE TRUTH, not at the defaults
    c_frozen = constrain(params["gp"])
    assert jnp.allclose(c_frozen["kappa"], cfg_disk["kappa"], atol=1e-5), \
        "globals frozen at the wrong kappa -> B(tau) misspecified"

    # 4) checkpoint round-trip (cfg above disables it, so exercise it explicitly)
    ckpt_cfg = TrainConfig(epochs=2, log_every=2, ckpt_every=1,
                           ckpt_path=str(CKPT_DIR / "_smoke_ckpt.pkl"))
    _, _ = train(loss_fn, params, batch, eval_, ckpt_cfg, net_d)
    with open(ckpt_cfg.ckpt_path, "rb") as f:
        ck = pickle.load(f)
    assert set(ck["params"]) == {"net_d", "net_A", "gp"}, "checkpoint pytree malformed"
    Path(ckpt_cfg.ckpt_path).unlink()

    # -----------------------------------------------------------------------
    # Gate readout. Under L_data alone the (delta, A) pair is only identified up
    # to  delta -> delta + c,  A -> A - c*B(tau):  both reproduce log F exactly.
    # So SHAPE is the recoverable quantity and LEVEL is not — the level offset is
    # expected to be nonzero until the NLL / A-ODE pins it (Phase 5/6).
    # -----------------------------------------------------------------------
    dh = net_d.apply(params["net_d"], eval_["t_grid"][:, None])[:, 0]
    dt = eval_["delta_true"]
    corr = float(jnp.corrcoef(dh, dt)[0, 1])
    off = float(jnp.mean(dh - dt))
    rmse_shape = float(jnp.sqrt(jnp.mean(((dh - jnp.mean(dh)) - (dt - jnp.mean(dt))) ** 2)))
    final_rmse_log = float(jnp.sqrt(hist[-1][1]))

    print(f"\n=== Phase 4 gate readout ===")
    print(f"  frozen-globals max move : {moved:.1e}")
    print(f"  frozen kappa            : {float(c_frozen['kappa']):.4f}  (true {cfg_disk['kappa']})")
    print(f"  batch rows P = (N+1)*K  : {P}")
    print(f"  L_data {hist[0][1]:.4e} -> {hist[-1][1]:.4e}")
    print(f"  rmse_log                : {final_rmse_log:.5f}  "
          f"(noise floor {mc_data['noise_std']})")
    print(f"  shape-corr(delta)       : {corr:+.4f}   <- the gate criterion")
    print(f"  de-meaned rmse(delta)   : {rmse_shape:.4f}")
    print(f"  level offset            : {off:+.4f}   (expected nonzero, unidentified)")
    print("HARNESS + L_DATA RUN COMPLETE")

=== mc_data.pkl ===
  paths M       : 100
  dates N+1     : 1001   (T = 10.0, N = 1000)
  maturities K  : 12     (tau 0.083 -> 1.000)
  noise_std     : 0.01
  params_P / lambda2 / alpha_Q all match config/*.json  ✓

=== batch (path 0) ===
  t     (12012,)   [ 0.0000,  10.0000]
  S     (12012,)   [ 35.8945,  165.1158]
  tau   (12012,)   [ 0.0833,  1.0000]
  logF  (12012,)   [ 3.5961,  5.0655]
  row order (date varies slow, maturity fast) verified  ✓
  log_price vs F_clean rmse : 2.84e-08  ✓
  implied A(tau) spread across dates : 1.56e-07  (should be ~0)

=== training: L_data (log-space MSE), globals frozen at truth ===
  target noise floor: rmse_log = noise_std = 0.01
epoch      0 | loss 9.5510e-02 | shape-corr +0.085 | level-off +0.0955
epoch    500 | loss 5.3157e-03 | shape-corr +0.476 | level-off -0.1276
epoch   1000 | loss 4.1054e-03 | shape-corr +0.632 | level-off -0.1336
epoch   1500 | loss 3.5873e-03 | shape-corr +0.691 | level-off -0.2216
epoch   2000 | loss 3.4850e-03 | shape-c